In [1]:
import picsure

In [2]:
my_token = ""

with open("token.txt") as f:
    my_token = f.read().strip()

auth_hpds_session = picsure.connect(
    picsure.Platform.BDC_AUTHORIZED,
    token=my_token,
    dev_mode=True
)

picsure.http /psama/user/me 200 355ms in=0B out=24.6KB retry=0
picsure.http /picsure/info/resources 200 134ms in=0B out=213B retry=0
picsure.http /psama/user/me/queryTemplate/ 200 117ms in=0B out=11.0KB retry=0
picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=1 200 141ms in=6.3KB out=635B retry=0
picsure.connect connect 0ms


You're successfully connected to BDC Authorized as user george_colon@hms.harvard.edu!
Your token expires on 2026-06-08T13:52:51Z.


In [3]:
facets = auth_hpds_session.facets()
facets.add("dataset_id", "phs000810")
facets.add("dataset_id", "phs000007")
facets.view()

picsure.http /picsure/proxy/dictionary-api/facets 200 147ms in=6.3KB out=116.9KB retry=0
picsure.fn session.facets 155ms


{'dataset_id': ['phs000810', 'phs000007'],
 'Consortium_Curated_Facets': [],
 'common_data_elements': [],
 'data_source': [],
 'data_type': []}

In [4]:
results = auth_hpds_session.searchDictionary("age", facets=facets)
results

picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=492262 200 226ms in=6.9KB out=1.6MB retry=0
picsure.fn session.searchDictionary 253ms


,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\phs000810\pht004715\phv00526256\AGE_IMMI\,phv00526256,AGE_IMMI,Age of immigration among participants who were...,Continuous,phs000810,[],0.0,73.0,True,None,HCHSSOL
1,\phs000007\pht000395\phv00056587\age_s1\,phv00056587,age_s1,Age = StdyDtqa - DOB,Continuous,phs000007,[],29.0,86.0,True,None,FHS
2,\phs000007\pht000397\phv00056723\age_s2\,phv00056723,age_s2,Age (age = formdate - DOB),Continuous,phs000007,[],44.0,86.0,True,None,FHS
3,\phs000007\pht003099\phv00177938\age5\,phv00177938,age5,Age at Exam 5,Continuous,phs000007,[],26.0,96.0,True,None,FHS
4,\phs000007\pht003099\phv00177948\age10\,phv00177948,age10,Age at Exam 10,Continuous,phs000007,[],46.0,102.0,True,None,FHS
...,...,...,...,...,...,...,...,...,...,...,...,...
2037,\phs000007\pht000676\phv00066829\phrm_gp1\,phv00066829,phrm_gp1,Description/Label for pharmacological subgroup...,Categorical,phs000007,"[ACE INHIBITORS, PLAIN, ADRENERGICS FOR SYSTEM...",NaN,NaN,False,None,FHS
2038,\phs000007\pht000676\phv00066827\system1\,phv00066827,system1,Description/Label for anatomical main group - ...,Categorical,phs000007,"[ALIMENTARY TRACT AND METABOLISM, ANTIINFECTIV...",NaN,NaN,False,None,FHS
2039,\phs000007\pht000676\phv00066823\phrm_gp2\,phv00066823,phrm_gp2,Description/Label for pharmacological subgroup...,Categorical,phs000007,"[ACE INHIBITORS, PLAIN, ANTACIDS, ANTIGLAUCOMA...",NaN,NaN,False,None,FHS
2040,\phs000007\pht000676\phv00066833\MEDNAME\,phv00066833,MEDNAME,Medication name,Categorical,phs000007,"[(ANTIOXIDANT), (ETHIN. ESTRADIO, (ETHINYL EST...",NaN,NaN,False,None,FHS


In [5]:
age5_phs000007 = results[results["display"] == "age5"]
age5_phs000007 = age5_phs000007[age5_phs000007["name"] == "phv00177938"]
age5_phs000007_clause = picsure.buildClause(
    age5_phs000007["conceptPath"].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    min=30,
    max=40
)

In [6]:
fhs_facet = auth_hpds_session.facets()
fhs_facet.add("dataset_id", "phs000007")
fhs_sex_results = auth_hpds_session.searchDictionary("phv00253990", facets=fhs_facet)
fhs_sex_results

picsure.http /picsure/proxy/dictionary-api/facets 200 150ms in=6.3KB out=116.9KB retry=0
picsure.fn session.facets 156ms
picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=492262 200 143ms in=6.6KB out=627B retry=0
picsure.fn session.searchDictionary 147ms


,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\phs000007\pht004374\phv00253990\sex\,phv00253990,sex,Sex of the participant,Categorical,phs000007,"[Female, Male]",None,None,True,None,FHS


In [7]:
fhs_sex_male_clause = picsure.buildClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Male"]
)

fhsMaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_male_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_sex_female_clause = picsure.buildClause(
    fhs_sex_results['conceptPath'].iloc[0],
    picsure.PhenotypicFilterType.FILTER,
    ["Female"]
)

fhsFemaleAnd30To40 = picsure.buildClauseGroup(
    [fhs_sex_female_clause, age5_phs000007_clause],
    operator=picsure.GroupOperator.AND
)

fhs_FemaleAges30to40_OR_MaleAges30to40 = picsure.buildClauseGroup(
    [fhsFemaleAnd30To40, fhsMaleAnd30To40],
    operator=picsure.GroupOperator.OR
)

fhs_FemaleAges30to40_OR_MaleAges30to40.to_query_json()

{'operator': 'OR',
 'phenotypicClauses': [{'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Female']},
    {'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht003099\\phv00177938\\age5\\',
     'not': False,
     'min': 30,
     'max': 40}],
   'not': False},
  {'operator': 'AND',
   'phenotypicClauses': [{'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht004374\\phv00253990\\sex\\',
     'not': False,
     'values': ['Male']},
    {'phenotypicFilterType': 'FILTER',
     'conceptPath': '\\phs000007\\pht003099\\phv00177938\\age5\\',
     'not': False,
     'min': 30,
     'max': 40}],
   'not': False}],
 'not': False}

In [8]:
auth_hpds_session.runQuery(picsure.buildQuery(phenotypicFilter=fhs_FemaleAges30to40_OR_MaleAges30to40))
# Results are within expected margin. Verified results with UI. UI Result: 77±3

picsure.http /picsure/v3/query/sync 200 10520ms in=826B out=2B retry=0
picsure.fn session.runQuery 10525ms


CountResult(value=80, margin=None, cap=None, raw='80')

In [9]:
# Persist a Query that both filters and selects output columns.
# includeConcepts is carried through save/load.
query_to_save = picsure.buildQuery(
    phenotypicFilter=fhs_FemaleAges30to40_OR_MaleAges30to40,
    includeConcepts=[
        age5_phs000007["conceptPath"].iloc[0],
        fhs_sex_results["conceptPath"].iloc[0],
    ],
)

query_id = auth_hpds_session.saveQueryByName(
    query_to_save,
    "Created from API adapters - fhs_FemaleAges30to40_OR_MaleAges30to40 - Test",
    overwrite=True,
)

picsure.http /picsure/dataset/named 200 210ms in=0B out=72.3KB retry=0
picsure.http /picsure/v3/query 200 160ms in=916B out=358B retry=0
picsure.http /picsure/dataset/named 200 174ms in=164B out=11.2KB retry=0
picsure.fn session.saveQueryByName 552ms


In [10]:
query = auth_hpds_session.loadQueryByID(query_id)

picsure.http /picsure/query/59ca33eb-6680-4bdf-b2c1-4b4aa40e4747/metadata 200 137ms in=0B out=8.9KB retry=0
picsure.fn session.loadQueryByID 142ms


In [11]:
auth_hpds_session.runQuery(query)

picsure.http /picsure/v3/query/sync 200 10370ms in=924B out=2B retry=0
picsure.fn session.runQuery 10372ms


CountResult(value=80, margin=None, cap=None, raw='80')